# Phase 1 Lab - Project Setup

Mục tiêu: kiểm tra skeleton của Shopping Assistant V3. Notebook này chạy trên
current codebase Phase 5A, nhưng chỉ focus vào những artifact Phase 1 tạo ra.

Expected output chính: `verify_setup: OK`.

Safety: không network, không secrets, không scraping, không model calls.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_v3_root(start: str | None = None) -> Path:
    path = Path(start or os.getcwd()).resolve()
    for candidate in (path, *path.parents):
        if candidate.name == "shopping_assistant_v3" and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the tech2ai/shopping_assistant_v3 tree.")


V3_ROOT = find_v3_root()
os.chdir(V3_ROOT)
if str(V3_ROOT) not in sys.path:
    sys.path.insert(0, str(V3_ROOT))


def run(command: list[str], timeout: int = 120) -> subprocess.CompletedProcess[str] | None:
    print("$ " + " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=V3_ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired:
        print(f"Command timed out after {timeout} seconds.")
        return None

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"exit_code={result.returncode}")
    return result


print(f"V3_ROOT={V3_ROOT}")
print("Default safety flags:")
for name in ("ENABLE_REAL_SEARCH", "ENABLE_REAL_MODEL_CALLS", "ENABLE_AGENTS_SDK"):
    print(f"{name}={os.getenv(name, '<unset>')}")


## 1. Chạy setup verification

Command này xác nhận các folder/file runtime tối thiểu tồn tại. Nếu có dòng
`MISSING DIR` hoặc `MISSING FILE`, các phase sau có thể không chạy đúng.


In [ ]:
run(["bash", "scripts/verify_setup.sh"])


## 2. Xem các folder quan trọng

Cell này không sửa file. Nó chỉ giúp bạn nhìn skeleton mà Phase 1 tạo ra.


In [ ]:
important_paths = [
    "backend",
    "backend/api",
    "backend/database",
    "backend/router",
    "backend/tools/deal_search",
    "backend/tools/price_estimator",
    "backend/synthesizer",
    "frontend",
    "scripts",
    ".env.example",
]

for item in important_paths:
    path = V3_ROOT / item
    kind = "dir" if path.is_dir() else "file" if path.is_file() else "missing"
    print(f"{item}: {kind}")


## 3. Cách đọc kết quả

- Tất cả dòng là `dir` hoặc `file`: skeleton đúng.
- `verify_setup: OK`: Phase 1 setup vẫn nguyên vẹn.
- Phase này chưa có backend API, database runtime, tools, hoặc frontend thật.
